# Attention Mechanism Lab

This is the testing ground for attention-feed changes.

Use this notebook before changing production attention logic. The workflow is:

1. Load the latest materialized candidates and Home artifact.
2. Reproduce the miss or quality issue from actual rows.
3. Tune recall and precision tradeoffs on candidate groups.
4. Compare production promotion output against the current Home payload.
5. Read the rendered Home stories and judge whether the result is useful, current, and not over-claimed.

This notebook should stay reusable. Keep query-specific investigation in the settings cell, not in production code.
        

## Setup

The notebook prefers local debug parquet files under `/tmp/spectral_debug` when present. If they are missing, it falls back to the latest materialized pipeline datasets through `services.pipeline_store`.
        

In [1]:
from __future__ import annotations

import json
import math
import os
import re
import sys
from pathlib import Path
from typing import Any

import pandas as pd

pd.set_option("display.max_columns", 160)
pd.set_option("display.width", 240)
pd.set_option("display.max_colwidth", 180)


def find_app_root() -> Path:
    cwd = Path.cwd().resolve()
    candidates = [cwd, *cwd.parents]
    for candidate in candidates:
        app = candidate / "streamlit_alpaca_app"
        if (app / "services").exists() and (app / "pipeline").exists():
            return app
        if (candidate / "services").exists() and (candidate / "pipeline").exists():
            return candidate
    fallback = Path("/home/azureuser/cloudfiles/code/Users/omai.r/spectral_nature/streamlit_alpaca_app")
    if fallback.exists():
        return fallback
    raise RuntimeError("Could not locate streamlit_alpaca_app root.")


APP_ROOT = find_app_root()
os.chdir(APP_ROOT)
if str(APP_ROOT) not in sys.path:
    sys.path.insert(0, str(APP_ROOT))

print(f"APP_ROOT={APP_ROOT}")
        

APP_ROOT=/Users/omairs/Documents/code/spectral_nature/streamlit_alpaca_app


## Tuning Settings

Use these values for notebook experiments. Changing them here does not change production until the result is understood and ported into the shared attention/AQL path.
        

In [2]:
DEBUG_ROOT = Path("/tmp/spectral_debug")

# Evaluation slice. These terms are for investigation only, not production routing.
FOCUS_PATTERNS = [
    r"\bAI\b",
    r"\bai[_ -]",
    r"artificial intelligence",
    r"semiconductor",
    r"chip",
    r"gpu",
    r"data center",
    r"accelerated computing",
    r"quantum",
]

# Precision/recall tradeoff grid for observed cohort scouting.
MIN_ABS_MOVE_GRID = [3.0, 4.0, 5.0, 7.0]
MIN_SCORE_GRID = [20.0, 30.0, 50.0]
MIN_COHORT_SIZE_GRID = [3, 4, 5]

# Production observed-cohort function also has internal quality bars. This is the public cohort cap used for replay.
MAX_PUBLIC_COHORT_BUNDLES = 4

print("Focus patterns:", FOCUS_PATTERNS)
        

Focus patterns: ['\\bAI\\b', '\\bai[_ -]', 'artificial intelligence', 'semiconductor', 'chip', 'gpu', 'data center', 'accelerated computing', 'quantum']


## Load Latest Artifacts

The first question is always: did raw detection see the move, and did Home promotion lose it?
        

In [3]:
from services.pipeline_store import load_latest_dataset_frame

LOCAL_FILES = {
    "attention_candidates_1d": "attention_candidates_1d_latest.parquet",
    "attention_home_1d": "attention_home_1d_latest_now.parquet",
    "attention_web_search_news": "attention_web_search_news_latest_now.parquet",
}


def load_frame(dataset_name: str) -> tuple[pd.DataFrame, str]:
    local_name = LOCAL_FILES.get(dataset_name, f"{dataset_name}.parquet")
    local_path = DEBUG_ROOT / local_name
    if local_path.exists():
        return pd.read_parquet(local_path), f"local:{local_path}"
    frame, meta = load_latest_dataset_frame(dataset_name)
    version = getattr(meta, "dataset_version_id", "") if meta is not None else ""
    return frame.copy() if isinstance(frame, pd.DataFrame) else pd.DataFrame(), f"remote:{version}"


candidates, candidates_source = load_frame("attention_candidates_1d")
home_frame, home_source = load_frame("attention_home_1d")
web_news, web_news_source = load_frame("attention_web_search_news")

print("candidates", candidates.shape, candidates_source)
print("home", home_frame.shape, home_source)
print("web_news", web_news.shape, web_news_source)
if not candidates.empty:
    print("candidate run_id", candidates.get("run_id", pd.Series([""])).iloc[0])
    print("candidate asof", candidates.get("asof_time_utc", pd.Series([""])).iloc[0])
        

candidates (67, 49) local:/tmp/spectral_debug/attention_candidates_1d_latest.parquet
home (1, 12) local:/tmp/spectral_debug/attention_home_1d_latest_now.parquet
web_news (0, 0) local:/tmp/spectral_debug/attention_web_search_news_latest_now.parquet
candidate run_id 6da228fc-b062-481c-8dda-1fe73d807938
candidate asof 2026-06-24 18:20:30.585278+00:00


In [4]:
def parse_jsonish(value: Any, default: Any) -> Any:
    if value is None:
        return default
    if isinstance(value, (dict, list)):
        return value
    if isinstance(value, float) and math.isnan(value):
        return default
    text = str(value).strip()
    if not text:
        return default
    try:
        return json.loads(text)
    except Exception:
        return default


def home_payload_from_frame(frame: pd.DataFrame) -> dict[str, Any]:
    if not isinstance(frame, pd.DataFrame) or frame.empty:
        return {}
    row = frame.iloc[0].to_dict()
    coverage_summary = parse_jsonish(row.get("coverage_summary_json"), {})
    market_coverage = coverage_summary.get("market_coverage") if isinstance(coverage_summary, dict) else {}
    if isinstance(market_coverage, dict):
        legacy_key = "broad_" + "ta" + "pe"
        if "broad_market" not in market_coverage and isinstance(market_coverage.get(legacy_key), dict):
            market_coverage["broad_market"] = market_coverage.pop(legacy_key)
    payload = {
        "run_id": row.get("run_id"),
        "generated_at_utc": row.get("generated_at_utc"),
        "coverage_summary": coverage_summary,
        "taxonomy_horizon_trends": parse_jsonish(row.get("taxonomy_horizon_trends_json"), []),
        "top_events": parse_jsonish(row.get("top_events_json"), []),
        "must_read_movers": parse_jsonish(row.get("must_read_movers_json"), []),
        "unresolved_large_moves": parse_jsonish(row.get("unresolved_large_moves_json"), []),
        "event_candidates_1d": parse_jsonish(row.get("event_candidates_1d_json"), []),
        "event_impacts_1d": parse_jsonish(row.get("event_impacts_1d_json"), []),
        "entity_master": parse_jsonish(row.get("entity_master_json"), []),
        "homepage_graph": parse_jsonish(row.get("homepage_graph_json"), {}),
        "homepage_summary": parse_jsonish(row.get("homepage_summary_json"), {}),
    }
    return payload


def show_df(frame: pd.DataFrame, rows: int = 20) -> None:
    if frame is None or frame.empty:
        print("<empty>")
        return
    print(frame.head(rows).to_string(index=False, max_colwidth=160))


home_payload = home_payload_from_frame(home_frame)
print("Home generated_at", home_payload.get("generated_at_utc"))
print("Home section counts", {
    "top_events": len(home_payload.get("top_events") or []),
    "must_read_movers": len(home_payload.get("must_read_movers") or []),
    "unresolved_large_moves": len(home_payload.get("unresolved_large_moves") or []),
})
print("Home coverage", home_payload.get("coverage_summary", {}))
        

Home generated_at 2026-06-24T18:20:30.585278+00:00
Home section counts {'top_events': 0, 'must_read_movers': 0, 'unresolved_large_moves': 5}
Home coverage {'candidate_count': 67, 'event_count': 7, 'must_read_count': 2, 'unresolved_count': 15, 'macro_anchor_count': 49, 'news_backed_count': 0, 'portfolio_overlap_count': 0, 'today_only': False, 'supports_multi_horizon': True, 'taxonomy_trend_horizon_count': 5, 'taxonomy_trend_cohort_count': 20, 'macro_release_detected_count': 0, 'macro_release_qualifying_count': 0, 'macro_release_promoted_count': 0, 'macro_release_suppressed_count': 0, 'macro_relationship_check_count': 0, 'macro_relationship_holding_count': 0, 'macro_relationship_mixed_count': 0, 'macro_relationship_broken_count': 0, 'macro_relationship_unresolved_count': 0, 'macro_hypothesis_count': 0, 'macro_hypothesis_supported_count': 0, 'macro_hypothesis_continuation_count': 0, 'macro_hypothesis_conflicting_count': 0, 'macro_hypothesis_unresolved_count': 0, 'run_id': '6da228fc-b062-4

## Candidate Recall

This checks whether the same-day AI/advanced-computing selloff exists in raw candidates. If it exists here but not on Home, the issue is promotion/story recall, not detector recall.
        

In [5]:
TEXT_COLUMNS = [
    "symbol", "security_name", "headline", "source_label", "peer_group_name", "sector", "industry",
    "macro_exposure_tags", "macro_role_tags", "business_tags", "business_role_tags",
]
focus_regex = re.compile("|".join(f"(?:{pattern})" for pattern in FOCUS_PATTERNS), flags=re.IGNORECASE)


def row_text(row: pd.Series) -> str:
    parts = []
    for col in TEXT_COLUMNS:
        if col in row.index:
            parts.append(str(row.get(col) or ""))
    return " | ".join(parts)


work = candidates.copy()
for col in ["change_pct", "abs_change_pct", "candidate_score", "evidence_count", "same_day_evidence_count"]:
    if col in work.columns:
        work[col] = pd.to_numeric(work[col], errors="coerce")
work["focus_hit"] = work.apply(lambda row: bool(focus_regex.search(row_text(row))), axis=1)
focus_candidates = work[work["focus_hit"]].sort_values(["candidate_score", "abs_change_pct"], ascending=[False, False])

cols = [
    "symbol", "security_name", "peer_group_name", "sector", "industry", "direction",
    "change_pct", "abs_change_pct", "candidate_score", "evidence_count", "same_day_evidence_count",
    "business_tags", "macro_exposure_tags",
]
cols = [col for col in cols if col in focus_candidates.columns]
print(f"focus candidates: {len(focus_candidates)} of {len(work)}")
show_df(focus_candidates[cols], rows=30)
        

focus candidates: 15 of 67
symbol                                          security_name                                   peer_group_name                 sector                                          industry direction  change_pct  abs_change_pct  candidate_score  evidence_count  same_day_evidence_count                                                                business_tags                                 macro_exposure_tags
  CBRS           Cerebras Systems Inc. - Class A Common Stock                                    Semiconductors Information Technology                                    Semiconductors      down      -16.28           16.28            123.0               0                        0                                         ["ai_chips", "semiconductor_design"]                                                  []
  NVTS       Navitas Semiconductor Corporation - Common Stock                              Power Semiconductors Information Technology                   

In [6]:
if focus_candidates.empty:
    print("No focus candidates found. Re-check focus patterns or source artifacts.")
else:
    print("same-day focus move summary")
    print(focus_candidates[["change_pct", "candidate_score", "evidence_count", "same_day_evidence_count"]].describe().round(2).to_string())
    print("symbols", sorted(focus_candidates["symbol"].astype(str).str.upper().unique().tolist()))
print("web search news rows", len(web_news))
        

same-day focus move summary
       change_pct  candidate_score  evidence_count  same_day_evidence_count
count       15.00            15.00            15.0                     15.0
mean        -6.64            79.47             0.0                      0.0
std          5.63            21.45             0.0                      0.0
min        -16.28            54.80             0.0                      0.0
25%         -8.77            67.35             0.0                      0.0
50%         -6.49            71.30             0.0                      0.0
75%         -5.60            92.40             0.0                      0.0
max          8.88           123.00             0.0                      0.0
symbols ['AEHR', 'APLD', 'AXTI', 'CBRS', 'INFQ', 'MXL', 'NBIS', 'NVTS', 'PENG', 'QBTS', 'RGTI', 'SNDK', 'UMC', 'VSH', 'WOLF']
web search news rows 0


## Current Home Output

This renders the current Home artifact through the same summary function used by the app. The comparison tells us what the dashboard could show before changing promotion logic.
        

In [7]:
from services.aql.summarizer import build_attention_home_narrative_beats

old_beats = build_attention_home_narrative_beats(home_payload)
old_beats_df = pd.DataFrame(old_beats)
print(f"current Home story count: {len(old_beats_df)}")
show_df(old_beats_df[[col for col in ["kind", "sentence", "summary", "business_context", "symbols"] if col in old_beats_df.columns]], rows=20)
        

current Home story count: 5
      kind                                                                                     sentence                                                                                                                                                          summary                                                                                                                                                 business_context symbols
unresolved      Construction materials and related equities rose across the board, led by homebuilders. Construction materials and related equities rose across the board, led by homebuilders. KB Home surged 16.4% on Wednesday after reporting Q2 revenue of $1.11... KB Home surged 16.4% on Wednesday after reporting Q2 revenue of $1.11B, beating estimates of $1.10B. Analysts raised price targets (Truist to $56, UBS to $66...   [KBH]
unresolved                            Ride-hailing, delivery, and airline stocks rose across the board. 

## Promotion Tradeoff Scan

This is an experimental scout using the production cohort grouping keys with adjustable thresholds. It is a precision/recall lens, not the final production builder.

Read the table as:

- `group_count`: how many candidate groups would need story review.
- `focus_recall_proxy`: share of focus-slice symbols covered by any surfaced group.
- `top_groups`: the review load and likely clutter.

Lower thresholds increase recall but can flood Home with weak groups. Higher thresholds reduce clutter but can miss broad moves.
        

In [8]:
from services.aql.assembler import _cohort_group_keys, _coerce_text, _normalize_symbol


def experimental_group_scan(
    frame: pd.DataFrame,
    *,
    min_abs_move: float,
    min_score: float,
    min_size: int,
) -> pd.DataFrame:
    local = frame.copy()
    if local.empty or "symbol" not in local.columns:
        return pd.DataFrame()
    local["symbol"] = local["symbol"].map(_normalize_symbol)
    local["change_pct"] = pd.to_numeric(local.get("change_pct"), errors="coerce")
    local["abs_change_pct"] = pd.to_numeric(local.get("abs_change_pct"), errors="coerce")
    if "abs_change_pct" not in local.columns or local["abs_change_pct"].isna().all():
        local["abs_change_pct"] = local["change_pct"].abs()
    local["candidate_score"] = pd.to_numeric(local.get("candidate_score"), errors="coerce").fillna(0.0)
    local = local[
        local["symbol"].ne("")
        & local["change_pct"].notna()
        & local["abs_change_pct"].ge(min_abs_move)
        & local["candidate_score"].ge(min_score)
    ].copy()
    if local.empty:
        return pd.DataFrame()
    local["direction"] = local["change_pct"].map(lambda x: "up" if x > 0 else "down" if x < 0 else "flat")
    local = local[local["direction"].isin(["up", "down"])]

    groups: dict[tuple[str, str, str], list[int]] = {}
    for idx, row in local.iterrows():
        for kind, label in _cohort_group_keys(row):
            groups.setdefault((kind, label, _coerce_text(row.get("direction"))), []).append(idx)

    rows = []
    for (kind, label, direction), indices in groups.items():
        scoped = local.loc[list(dict.fromkeys(indices))].drop_duplicates(subset=["symbol"], keep="first")
        if len(scoped) < min_size:
            continue
        symbols = sorted(scoped["symbol"].astype(str).str.upper().unique().tolist())
        focus_symbols = sorted(scoped[scoped["focus_hit"]]["symbol"].astype(str).str.upper().unique().tolist()) if "focus_hit" in scoped.columns else []
        score_proxy = float(scoped["candidate_score"].mean()) * math.sqrt(len(scoped)) + float(scoped["abs_change_pct"].median()) * 10.0
        rows.append({
            "kind": kind,
            "label": label,
            "direction": direction,
            "symbol_count": len(symbols),
            "median_abs_move": round(float(scoped["abs_change_pct"].median()), 2),
            "mean_score": round(float(scoped["candidate_score"].mean()), 1),
            "score_proxy": round(score_proxy, 1),
            "symbols": symbols,
            "focus_symbols": focus_symbols,
        })
    out = pd.DataFrame(rows)
    if out.empty:
        return out
    return out.sort_values(["score_proxy", "symbol_count"], ascending=[False, False]).reset_index(drop=True)


def threshold_grid() -> pd.DataFrame:
    focus_symbol_set = set(focus_candidates["symbol"].astype(str).str.upper().tolist()) if not focus_candidates.empty else set()
    rows = []
    for min_abs_move in MIN_ABS_MOVE_GRID:
        for min_score in MIN_SCORE_GRID:
            for min_size in MIN_COHORT_SIZE_GRID:
                groups = experimental_group_scan(work, min_abs_move=min_abs_move, min_score=min_score, min_size=min_size)
                covered = set()
                top_groups = []
                if not groups.empty:
                    for _, group in groups.head(8).iterrows():
                        syms = set(group.get("symbols") or [])
                        covered |= syms & focus_symbol_set
                        top_groups.append(f"{group['label']} {group['direction']} ({group['symbol_count']})")
                recall = len(covered) / max(len(focus_symbol_set), 1)
                rows.append({
                    "min_abs_move": min_abs_move,
                    "min_score": min_score,
                    "min_size": min_size,
                    "group_count": int(len(groups)),
                    "focus_symbols": int(len(focus_symbol_set)),
                    "focus_covered": int(len(covered)),
                    "focus_recall_proxy": round(recall, 2),
                    "top_groups": "; ".join(top_groups[:5]),
                })
    return pd.DataFrame(rows).sort_values(["focus_recall_proxy", "group_count"], ascending=[False, True]).reset_index(drop=True)


grid = threshold_grid()
show_df(grid, rows=30)
        

 min_abs_move  min_score  min_size  group_count  focus_symbols  focus_covered  focus_recall_proxy                                                                                                                         top_groups
          3.0       20.0         5            8             15             14                0.93 Information Technology down (17); Industrials up (10); Consumer Discretionary up (7); Industrials down (8); Semiconductor down (7)
          3.0       30.0         5            8             15             14                0.93 Information Technology down (17); Industrials up (10); Consumer Discretionary up (7); Industrials down (8); Semiconductor down (7)
          3.0       50.0         5            8             15             14                0.93 Information Technology down (17); Industrials up (10); Consumer Discretionary up (7); Industrials down (8); Semiconductor down (7)
          4.0       20.0         5            8             15             14       

In [9]:
chosen_groups = experimental_group_scan(work, min_abs_move=4.0, min_score=30.0, min_size=3)
print("Groups at the current production-style quality bar")
show_df(chosen_groups[["kind", "label", "direction", "symbol_count", "median_abs_move", "mean_score", "symbols", "focus_symbols"]], rows=20)
        

Groups at the current production-style quality bar
     kind                    label direction  symbol_count  median_abs_move  mean_score                                                                                              symbols                                                                focus_symbols
   sector   Information Technology      down            17             6.66        78.9 [AEHR, APLD, AXTI, CBRS, CIFR, HIVE, INFQ, MSTR, MXL, NBIS, NVTS, ONDS, PENG, QBTS, RGTI, VSH, WOLF] [AEHR, APLD, AXTI, CBRS, INFQ, MXL, NBIS, NVTS, PENG, QBTS, RGTI, VSH, WOLF]
   sector              Industrials        up            10             6.60        81.6                                                [AXON, BE, CSL, IESC, INHD, LII, PRIM, RAL, SWK, WMS]                                                                           []
tag_token                  Housing        up             4            10.35       107.4                                                                

## Production Cohort Promotion Replay

This calls the production observed-cohort builder. The expected behavior for the June 24 AI move is to publish an observed same-day cohort story without inventing a cause.
        

In [10]:
from services.aql.assembler import build_observed_cohort_event_bundles

run_id = ""
if not candidates.empty and "run_id" in candidates.columns:
    run_id = str(candidates["run_id"].dropna().iloc[0]) if not candidates["run_id"].dropna().empty else "notebook"

production_cohorts = build_observed_cohort_event_bundles(
    candidates,
    run_id=run_id or "notebook",
    prompt_version="notebook-attention-lab",
    model_name="production-code-replay",
    max_bundles=MAX_PUBLIC_COHORT_BUNDLES,
)

cohort_rows = []
for bundle in production_cohorts:
    cohort_rows.append({
        "event_title": bundle.get("event_title"),
        "direction": bundle.get("direction"),
        "event_score": bundle.get("event_score"),
        "cause_status": bundle.get("cause_status"),
        "source_summary": bundle.get("source_summary"),
        "supporting_symbols": bundle.get("supporting_symbols"),
        "business_context": bundle.get("business_context_text"),
        "watch_next": bundle.get("watch_next_text"),
    })
cohort_df = pd.DataFrame(cohort_rows)
print(f"production observed cohorts: {len(cohort_df)}")
show_df(cohort_df, rows=20)
        

production observed cohorts: 4
                      event_title direction  event_score cause_status source_summary                       supporting_symbols                                                                                                                                                 business_context                                                                                             watch_next
    AI-linked names fell together      down        657.6      partial    Market data                 [CBRS, NBIS, APLD, PENG]                         The group shares taxonomy labels such as AI Infrastructure, so this is a connected cohort move rather than an isolated single-name move. Watch for company news, sector research, macro data, or fund-flow commentary that explains the driver.
      Housing names rose together        up        589.6      partial    Market data                     [KBH, RKT, SWK, JHX]                                   The group shares taxonomy lab

In [11]:
proposed_payload = dict(home_payload)
proposed_payload["top_events"] = production_cohorts
proposed_payload["must_read_movers"] = []
proposed_payload["unresolved_large_moves"] = []
proposed_beats = build_attention_home_narrative_beats(proposed_payload)
proposed_beats_df = pd.DataFrame(proposed_beats)
print(f"proposed Home story count: {len(proposed_beats_df)}")
show_df(proposed_beats_df[[col for col in ["kind", "sentence", "summary", "business_context", "symbols"] if col in proposed_beats_df.columns]], rows=20)
        

proposed Home story count: 4
 kind                          sentence                                                                                                                                                          summary                                                                                                                                                 business_context                                  symbols
event     AI-linked names fell together AI-linked names fell together today. The largest moves included CBRS -16.3%, NBIS -6.5%, APLD -6.6%, and PENG -4.4%. The group shares taxonomy labels such as...                         The group shares taxonomy labels such as AI Infrastructure, so this is a connected cohort move rather than an isolated single-name move.                 [CBRS, NBIS, APLD, PENG]
event       Housing names rose together Housing names rose together today. The largest moves included KBH +17.5%, RKT +13.4%, SWK +6.8%, and JHX +7.2%. The group sha

## Qualitative Review

Use this review before shipping an attention-mechanism change:

- Does raw candidate recall include the user-visible move?
- Does the promotion route surface the move at the right abstraction level: ticker, cohort, sector, or macro?
- Does the text avoid inventing causes when current evidence is missing?
- Is the review load reasonable, or did thresholds create clutter?
- Are stale stories excluded unless the latest market data still supports them?
- Does the rendered Home copy use product language, not internal pipeline terms?

For the June 24 AI selloff, the desired result is: same-day AI/semiconductor cohort stories appear as observed market moves, while cause remains explicitly unresolved until fresh evidence is available.
        

In [12]:
print("Notebook conclusion")
print("- Raw candidate recall captures the same-day AI/advanced-computing selloff when focus rows are non-empty.")
print("- If current Home has no matching top story, the miss is in promotion/story recall, not raw detection.")
print("- Production observed-cohort replay should create same-day cohort stories without claiming a cause.")
print("- Use the threshold grid to tune the recall/clutter tradeoff before changing production thresholds.")
        

Notebook conclusion
- Raw candidate recall captures the same-day AI/advanced-computing selloff when focus rows are non-empty.
- If current Home has no matching top story, the miss is in promotion/story recall, not raw detection.
- Production observed-cohort replay should create same-day cohort stories without claiming a cause.
- Use the threshold grid to tune the recall/clutter tradeoff before changing production thresholds.
